# Compare Original Folder vs Organized Folder

Goal: check whether the organized folder is inconsistent with the original folder.

This notebook reports the exact files causing inconsistency:
- missing from organized
- extra in organized
- same filename but content changed
- duplicates
- zero-byte files
- extension/name conflicts


In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from datetime import datetime

# =========================
# EDIT THESE THREE PATHS
# =========================
ORIGINAL_DIR = Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5')
ORGANIZED_DIR = Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy')
REPORT_DIR = Path(r"/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".heic"}
RECURSIVE = True

# "filename" is best when the organized folder uses category subfolders.
# "relative_path" is best when both folder structures should be identical.
MATCH_MODE = "filename"

# True confirms exact byte-for-byte copy.
COMPUTE_HASH = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Original:", ORIGINAL_DIR)
print("Organized:", ORGANIZED_DIR)
print("Reports:", REPORT_DIR)


Original: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5
Organized: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy
Reports: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports


In [2]:
def iter_image_files(root: Path, recursive=True):
    pattern = "**/*" if recursive else "*"
    for p in root.glob(pattern):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_key(path: Path, root: Path, mode: str):
    if mode == "relative_path":
        return str(path.relative_to(root)).replace("\\", "/")
    if mode == "filename":
        return path.name
    raise ValueError("MATCH_MODE must be 'filename' or 'relative_path'")

def inventory(root: Path, side_name: str):
    rows = []
    files = list(iter_image_files(root, RECURSIVE))
    print(f"{side_name}: found {len(files)} image files")
    for i, p in enumerate(files, 1):
        stat = p.stat()
        rows.append({
            "side": side_name,
            "key": make_key(p, root, MATCH_MODE),
            "filename": p.name,
            "stem_lower": p.stem.lower(),
            "extension": p.suffix.lower(),
            "relative_path": str(p.relative_to(root)).replace("\\", "/"),
            "absolute_path": str(p.resolve()),
            "size_bytes": stat.st_size,
            "mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
            "sha256": sha256_file(p) if COMPUTE_HASH else "",
            "zero_bytes": stat.st_size == 0,
        })
        if i % 500 == 0:
            print(f"  scanned {i}/{len(files)}")
    return pd.DataFrame(rows)

orig = inventory(ORIGINAL_DIR, "original")
org = inventory(ORGANIZED_DIR, "organized")
all_inventory = pd.concat([orig, org], ignore_index=True)
all_inventory.to_csv(REPORT_DIR / "all_inventory.csv", index=False)
display(all_inventory.head())
print("Saved:", REPORT_DIR / "all_inventory.csv")


original: found 1007 image files
  scanned 500/1007
  scanned 1000/1007
organized: found 1007 image files
  scanned 500/1007
  scanned 1000/1007


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes
0,original,20220214154603_pid2625_enIMWE.jpeg,20220214154603_pid2625_enIMWE.jpeg,20220214154603_pid2625_enimwe,.jpeg,20220214154603_pid2625_enIMWE.jpeg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1328911,2023-07-13T01:04:37,d569d351f3139df8aac19c1879ddcefdd2373d72ab0592...,False
1,original,20221229094001_pid2625_8fgoxu.jpeg,20221229094001_pid2625_8fgoxu.jpeg,20221229094001_pid2625_8fgoxu,.jpeg,20221229094001_pid2625_8fgoxu.jpeg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1551315,2023-07-13T01:15:30,a36b7ee6d347f76a986bb434629d0bfca2076ab2bf351d...,False
2,original,20221104140803_pid2625_iv6Sww.jpeg,20221104140803_pid2625_iv6Sww.jpeg,20221104140803_pid2625_iv6sww,.jpeg,20221104140803_pid2625_iv6Sww.jpeg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1315135,2023-07-13T01:12:31,1aa54d45b89ea95cdd74a2d2af144ab8b0ae4834bfd8d1...,False
3,original,20220627144201_pid2625_LjpvEi.jpeg,20220627144201_pid2625_LjpvEi.jpeg,20220627144201_pid2625_ljpvei,.jpeg,20220627144201_pid2625_LjpvEi.jpeg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,2045275,2023-07-13T01:09:10,b1529ace68da15d18212c2382f9cda75116c3e22353a50...,False
4,original,20210608153850_pid2625_cjvJU8.jpg,20210608153850_pid2625_cjvJU8.jpg,20210608153850_pid2625_cjvju8,.jpg,20210608153850_pid2625_cjvJU8.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1124436,2023-07-13T00:51:38,b425a57eec9cc7f7f5ca3ffab18de152e25982417f7d21...,False


Saved: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/all_inventory.csv


In [3]:
def save_df(df, name):
    path = REPORT_DIR / name
    df.to_csv(path, index=False)
    print(f"{name}: {len(df)} rows -> {path}")
    return path

# Duplicate matching keys
dup_key_orig = orig[orig.duplicated("key", keep=False)].sort_values("key")
dup_key_org = org[org.duplicated("key", keep=False)].sort_values("key")
save_df(dup_key_orig, "duplicate_match_keys_original.csv")
save_df(dup_key_org, "duplicate_match_keys_organized.csv")

if len(dup_key_orig) or len(dup_key_org):
    print("WARNING: duplicate match keys found. If MATCH_MODE='filename', repeated filenames exist in subfolders.")


duplicate_match_keys_original.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/duplicate_match_keys_original.csv
duplicate_match_keys_organized.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/duplicate_match_keys_organized.csv


In [4]:
# Missing / extra by key
orig_keys = set(orig["key"])
org_keys = set(org["key"])

missing = orig[orig["key"].isin(sorted(orig_keys - org_keys))].sort_values("key")
extra = org[org["key"].isin(sorted(org_keys - orig_keys))].sort_values("key")

save_df(missing, "missing_in_organized.csv")
save_df(extra, "extra_in_organized.csv")

display(missing.head(20))
display(extra.head(20))


missing_in_organized.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/missing_in_organized.csv
extra_in_organized.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/extra_in_organized.csv


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes


In [5]:
# Same key but changed content
orig_unique = orig.drop_duplicates("key", keep=False)
org_unique = org.drop_duplicates("key", keep=False)
merged = orig_unique.merge(org_unique, on="key", suffixes=("_original", "_organized"), how="inner")

if COMPUTE_HASH:
    mismatch = merged[merged["sha256_original"] != merged["sha256_organized"]].copy()
else:
    mismatch = merged[merged["size_bytes_original"] != merged["size_bytes_organized"]].copy()

def reason(row):
    reasons = []
    if row["size_bytes_original"] != row["size_bytes_organized"]:
        reasons.append("size_changed")
    if COMPUTE_HASH and row["sha256_original"] != row["sha256_organized"]:
        reasons.append("hash_changed")
    if row["extension_original"] != row["extension_organized"]:
        reasons.append("extension_changed")
    return ";".join(reasons) if reasons else "same"

if len(mismatch):
    mismatch["inconsistency_reason"] = mismatch.apply(reason, axis=1)
else:
    mismatch["inconsistency_reason"] = []

save_df(mismatch, "same_name_but_content_changed.csv")
cols = [c for c in [
    "key", "inconsistency_reason", "relative_path_original", "relative_path_organized",
    "size_bytes_original", "size_bytes_organized", "sha256_original", "sha256_organized"
] if c in mismatch.columns]
display(mismatch[cols].head(30) if len(mismatch) else mismatch)


same_name_but_content_changed.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/same_name_but_content_changed.csv


,side_original,key,filename_original,stem_lower_original,extension_original,relative_path_original,absolute_path_original,size_bytes_original,mtime_original,sha256_original,...,filename_organized,stem_lower_organized,extension_organized,relative_path_organized,absolute_path_organized,size_bytes_organized,mtime_organized,sha256_organized,zero_bytes_organized,inconsistency_reason


In [6]:
# Duplicate file content by hash
def duplicate_hashes(df):
    if not COMPUTE_HASH or "sha256" not in df.columns:
        return pd.DataFrame()
    valid = df[df["sha256"] != ""].copy()
    return valid[valid.duplicated("sha256", keep=False)].sort_values(["sha256", "relative_path"])

dup_hash_orig = duplicate_hashes(orig)
dup_hash_org = duplicate_hashes(org)
save_df(dup_hash_orig, "duplicate_file_content_original.csv")
save_df(dup_hash_org, "duplicate_file_content_organized.csv")
display(dup_hash_org.head(30))


duplicate_file_content_original.csv: 49 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/duplicate_file_content_original.csv
duplicate_file_content_organized.csv: 49 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/duplicate_file_content_organized.csv


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes
706,organized,20220630143133_pid2625_VBpKMR.jpeg,20220630143133_pid2625_VBpKMR.jpeg,20220630143133_pid2625_vbpkmr,.jpeg,usable/usable_images/20220630143133_pid2625_VB...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1424480,2023-07-13T01:09:39,5dcc2ef562f404b02e29cb6a41e601d22edfea1dd6e597...,False
616,organized,20220630143932_pid2625_VzYXoh.jpeg,20220630143932_pid2625_VzYXoh.jpeg,20220630143932_pid2625_vzyxoh,.jpeg,usable/usable_images/20220630143932_pid2625_Vz...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1424480,2023-07-13T01:09:41,5dcc2ef562f404b02e29cb6a41e601d22edfea1dd6e597...,False
449,organized,20210615170630_pid2625_2JMAqH.jpeg,20210615170630_pid2625_2JMAqH.jpeg,20210615170630_pid2625_2jmaqh,.jpeg,usable/usable_images/20210615170630_pid2625_2J...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1509177,2023-07-13T00:52:08,a4fdf9e7af04cd894f61b3e862a10c6be4ad9adb0702d2...,False
840,organized,20210615183303_pid2625_hMdXyL.jpeg,20210615183303_pid2625_hMdXyL.jpeg,20210615183303_pid2625_hmdxyl,.jpeg,usable/usable_images/20210615183303_pid2625_hM...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1509177,2023-07-13T00:52:09,a4fdf9e7af04cd894f61b3e862a10c6be4ad9adb0702d2...,False
417,organized,20210727171359_pid2625_GJcoYc.jpeg,20210727171359_pid2625_GJcoYc.jpeg,20210727171359_pid2625_gjcoyc,.jpeg,usable/usable_images/20210727171359_pid2625_GJ...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1520255,2023-07-13T00:56:33,d3c3addf991cbcde6fb92cee7b4e118789e059fb099a60...,False
461,organized,20210727172943_pid2625_vnS9Cf.jpeg,20210727172943_pid2625_vnS9Cf.jpeg,20210727172943_pid2625_vns9cf,.jpeg,usable/usable_images/20210727172943_pid2625_vn...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,1520255,2023-07-13T00:56:35,d3c3addf991cbcde6fb92cee7b4e118789e059fb099a60...,False
19,organized,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_uhnuf5,.jpg,anomaly/zero_bytes/20210628172455_pid2625_UHnu...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
39,organized,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gch2yt,.jpg,anomaly/zero_bytes/20210628173329_pid2625_gCH2...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
46,organized,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_cqr2k2,.jpg,anomaly/zero_bytes/20210628174514_pid2625_CqR2...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
15,organized,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sianip,.jpg,anomaly/zero_bytes/20210629122158_pid2625_sIaN...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True


In [7]:
# Same stem but different extensions
def stem_conflicts(df):
    if len(df) == 0:
        return df.copy()
    ext_counts = df.groupby("stem_lower")["extension"].nunique()
    conflict_stems = ext_counts[ext_counts > 1].index
    return df[df["stem_lower"].isin(conflict_stems)].sort_values(["stem_lower", "extension"])

stem_conflict_orig = stem_conflicts(orig)
stem_conflict_org = stem_conflicts(org)
save_df(stem_conflict_orig, "same_stem_different_extensions_original.csv")
save_df(stem_conflict_org, "same_stem_different_extensions_organized.csv")
display(stem_conflict_org.head(30))


same_stem_different_extensions_original.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/same_stem_different_extensions_original.csv
same_stem_different_extensions_organized.csv: 0 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/same_stem_different_extensions_organized.csv


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes


In [8]:
# Zero-byte files
zero_orig = orig[orig["zero_bytes"]].sort_values("relative_path")
zero_org = org[org["zero_bytes"]].sort_values("relative_path")
save_df(zero_orig, "zero_byte_files_original.csv")
save_df(zero_org, "zero_byte_files_organized.csv")
display(zero_orig.head(20))
display(zero_org.head(20))


zero_byte_files_original.csv: 43 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/zero_byte_files_original.csv
zero_byte_files_organized.csv: 43 rows -> /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/folder_compare_reports/zero_byte_files_organized.csv


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes
344,original,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_uhnuf5,.jpg,20210628172455_pid2625_UHnuF5.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
843,original,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gch2yt,.jpg,20210628173329_pid2625_gCH2YT.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
925,original,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_cqr2k2,.jpg,20210628174514_pid2625_CqR2k2.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
247,original,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sianip,.jpg,20210629122158_pid2625_sIaNIp.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
466,original,20210629123853_pid2625_tBnRpI.jpg,20210629123853_pid2625_tBnRpI.jpg,20210629123853_pid2625_tbnrpi,.jpg,20210629123853_pid2625_tBnRpI.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
297,original,20210629141113_pid2625_ibkCqo.jpg,20210629141113_pid2625_ibkCqo.jpg,20210629141113_pid2625_ibkcqo,.jpg,20210629141113_pid2625_ibkCqo.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
80,original,20210629145732_pid2625_985urE.jpg,20210629145732_pid2625_985urE.jpg,20210629145732_pid2625_985ure,.jpg,20210629145732_pid2625_985urE.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
911,original,20210629163035_pid2625_o2YgHV.jpg,20210629163035_pid2625_o2YgHV.jpg,20210629163035_pid2625_o2yghv,.jpg,20210629163035_pid2625_o2YgHV.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
263,original,20210629164800_pid2625_nKx4RE.jpg,20210629164800_pid2625_nKx4RE.jpg,20210629164800_pid2625_nkx4re,.jpg,20210629164800_pid2625_nKx4RE.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
934,original,20210629170240_pid2625_TYiZFY.jpg,20210629170240_pid2625_TYiZFY.jpg,20210629170240_pid2625_tyizfy,.jpg,20210629170240_pid2625_TYiZFY.jpg,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True


,side,key,filename,stem_lower,extension,relative_path,absolute_path,size_bytes,mtime,sha256,zero_bytes
19,organized,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_UHnuF5.jpg,20210628172455_pid2625_uhnuf5,.jpg,anomaly/zero_bytes/20210628172455_pid2625_UHnu...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
39,organized,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gCH2YT.jpg,20210628173329_pid2625_gch2yt,.jpg,anomaly/zero_bytes/20210628173329_pid2625_gCH2...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
46,organized,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_CqR2k2.jpg,20210628174514_pid2625_cqr2k2,.jpg,anomaly/zero_bytes/20210628174514_pid2625_CqR2...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
15,organized,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sIaNIp.jpg,20210629122158_pid2625_sianip,.jpg,anomaly/zero_bytes/20210629122158_pid2625_sIaN...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
26,organized,20210629123853_pid2625_tBnRpI.jpg,20210629123853_pid2625_tBnRpI.jpg,20210629123853_pid2625_tbnrpi,.jpg,anomaly/zero_bytes/20210629123853_pid2625_tBnR...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
17,organized,20210629141113_pid2625_ibkCqo.jpg,20210629141113_pid2625_ibkCqo.jpg,20210629141113_pid2625_ibkcqo,.jpg,anomaly/zero_bytes/20210629141113_pid2625_ibkC...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
10,organized,20210629145732_pid2625_985urE.jpg,20210629145732_pid2625_985urE.jpg,20210629145732_pid2625_985ure,.jpg,anomaly/zero_bytes/20210629145732_pid2625_985u...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
44,organized,20210629163035_pid2625_o2YgHV.jpg,20210629163035_pid2625_o2YgHV.jpg,20210629163035_pid2625_o2yghv,.jpg,anomaly/zero_bytes/20210629163035_pid2625_o2Yg...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
16,organized,20210629164800_pid2625_nKx4RE.jpg,20210629164800_pid2625_nKx4RE.jpg,20210629164800_pid2625_nkx4re,.jpg,anomaly/zero_bytes/20210629164800_pid2625_nKx4...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True
47,organized,20210629170240_pid2625_TYiZFY.jpg,20210629170240_pid2625_TYiZFY.jpg,20210629170240_pid2625_tyizfy,.jpg,anomaly/zero_bytes/20210629170240_pid2625_TYiZ...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,0,2023-07-13T00:53:25,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True


In [9]:
summary = {
    "original_image_count": int(len(orig)),
    "organized_image_count": int(len(org)),
    "missing_in_organized_count": int(len(missing)),
    "extra_in_organized_count": int(len(extra)),
    "same_key_but_content_changed_count": int(len(mismatch)),
    "duplicate_match_keys_original_count": int(len(dup_key_orig)),
    "duplicate_match_keys_organized_count": int(len(dup_key_org)),
    "duplicate_same_content_original_count": int(len(dup_hash_orig)),
    "duplicate_same_content_organized_count": int(len(dup_hash_org)),
    "zero_byte_original_count": int(len(zero_orig)),
    "zero_byte_organized_count": int(len(zero_org)),
    "match_mode": MATCH_MODE,
    "recursive": RECURSIVE,
    "computed_hash": COMPUTE_HASH,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(REPORT_DIR / "SUMMARY_folder_consistency.csv", index=False)
(REPORT_DIR / "SUMMARY_folder_consistency.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(summary_df)

if (
    summary["missing_in_organized_count"] == 0 and
    summary["extra_in_organized_count"] == 0 and
    summary["same_key_but_content_changed_count"] == 0 and
    summary["duplicate_match_keys_original_count"] == 0 and
    summary["duplicate_match_keys_organized_count"] == 0
):
    print("✅ Main check passed: organized folder is consistent with original folder.")
else:
    print("⚠️ Inconsistency found. Open the CSV reports to see exact image paths.")


,original_image_count,organized_image_count,missing_in_organized_count,extra_in_organized_count,same_key_but_content_changed_count,duplicate_match_keys_original_count,duplicate_match_keys_organized_count,duplicate_same_content_original_count,duplicate_same_content_organized_count,zero_byte_original_count,zero_byte_organized_count,match_mode,recursive,computed_hash
0,1007,1007,0,0,0,0,0,49,49,43,43,filename,True,True


✅ Main check passed: organized folder is consistent with original folder.


## How to read the reports

Most important reports:

- `SUMMARY_folder_consistency.csv`: one-row summary.
- `missing_in_organized.csv`: images in original but not in organized.
- `extra_in_organized.csv`: images in organized but not in original.
- `same_name_but_content_changed.csv`: same filename/key exists in both, but content hash changed.
- `duplicate_match_keys_*.csv`: repeated filenames/keys that make comparison ambiguous.
- `duplicate_file_content_organized.csv`: duplicate copied images in the organized folder.
- `zero_byte_files_*.csv`: broken empty files.

For category folders like `usable/`, `unusable/`, `anomaly/`, use:

```python
MATCH_MODE = "filename"
RECURSIVE = True
```

For checking exact same folder structure, use:

```python
MATCH_MODE = "relative_path"
RECURSIVE = True
```
